In [ ]:
import polars as pl

### Resumen del capítulo: Visualización de datos

En este capítulo aprenderás a:

- Crear rápidamente gráficos de barras, dispersión, densidad e histogramas usando Polars y Altair.
- Componer y superponer múltiples gráficos.
- Crear visualizaciones interactivas.
- Visualizar millones de puntos en mapas.
- Utilizar paquetes alternativos de visualización como hvPlot y plotnine.
- Crear tablas visualmente atractivas con Great Tables.

Al finalizar, sabrás qué ofrecen Altair, hvPlot, plotnine y Great Tables, cuándo usar cada uno y cómo integrarlos con Polars para análisis y visualización de datos.

## NYC Bike Trips

In [ ]:
trips = pl.read_parquet("data/citibike/*.parquet")
print(trips[:, :4])
print(trips[:, 4:7])
print(trips[:, 7:11])
print(trips[:, 11:])

In [ ]:
import altair as alt

| Namespace         | Método                | Descripción                                                                                   |
|-------------------|-----------------------|----------------------------------------------------------------------------------------------|
| `df.plot`         | `bar()`               | Dibuja un gráfico de barras, puede ser apilado o agrupado.                                   |
| `df.plot`         | `line()`              | Dibuja un gráfico de líneas, útil para series temporales.                                    |
| `df.plot`         | `point()`             | Dibuja un diagrama de dispersión (scatter) para comparar dos variables.                      |
| `df.plot`         | `scatter()`           | Alias de `point()`, también crea un diagrama de dispersión.                                  |
| `series.plot`     | `hist()`              | Dibuja la distribución de una o más variables como histogramas (conjunto de bins).           |
| `series.plot`     | `density()`           | Dibuja la estimación de densidad kernel (KDE) de una o más variables.                        |
| `series.plot`     | `line()`              | Dibuja un gráfico de líneas para una serie temporal o secuencia de datos.                    |

## Plotting DataFrames

In [ ]:
trips_speed  = trips.select(
    pl.col('distance'),
    pl.col('duration').dt.total_seconds() / 3600.0,
    pl.col('bike_type'),
).with_columns(
    (pl.col('distance') / pl.col('duration')).alias('speed')
)

trips_speed

In [ ]:
trips_speed.describe()

In [ ]:
trips_speed.plot.scatter(
    x="distance",
    y="duration",
    color="bike_type",
)

In [ ]:
trips_speed = (
    trips.filter(pl.col("station_start") == "W 70 St & Amsterdam Ave")
    .select(
        pl.col("distance"),
        pl.col("duration").dt.total_seconds() / 3600,
        pl.col("bike_type"),
    )
    .with_columns(speed=pl.col("distance") / pl.col("duration"))
)
trips_speed

In [ ]:
trips_speed.plot.scatter(
    x="distance",
    y="duration",
    color="bike_type",  
)

##  Too Large to Handle

In [ ]:
import altair as alt
alt.data_transformers.disable_max_rows()

In [ ]:
alt.data_transformers.enable("vegafusion")

In [ ]:
trips_type_counts = trips.group_by("rider_type", "bike_type").len()
trips_type_counts

In [ ]:
trips_type_counts.plot.bar(
    x="rider_type", 
    y="len", 
    by="bike_type", # 'by' creates the groups/colors
    stacked=True
).opts(
    width=500 # .opts replaces .properties
)

## Plotting Series

In [ ]:
trips_speed["duration"].plot.kde()

In [ ]:
trips_speed["distance"].plot.hist()